## Example 1: Basic File Path and Name Generation

In [ ]:
from faker import Faker
from faker_file.providers.txt_file import TxtFileProvider
from faker_file.storages.filesystem import FileSystemStorage
import os
import tempfile

# Example 1: Generate real files with faker_file, plus file names and paths
fake = Faker()
fake.add_provider(TxtFileProvider)

print("=== Example 1: Basic File Path and Name Generation ===")

try:
    with tempfile.TemporaryDirectory() as tmpdir:
        # FileSystemStorage writes under root_path/rel_path; a temporary root keeps the run self-cleaning
        storage = FileSystemStorage(root_path=tmpdir, rel_path="generated")

        print("Generated TXT files:")
        for i in range(3):
            txt = fake.txt_file(storage=storage, max_nb_chars=200)
            path = txt.data["filename"]  # absolute path; the returned value is the path relative to root_path
            assert os.path.exists(path), path
            assert path.endswith(".txt") and os.path.dirname(path) == os.path.join(tmpdir, "generated"), path
            assert txt == os.path.relpath(path, tmpdir), (txt, path)
            with open(path, encoding="utf-8") as fh:
                content = fh.read()
            assert content == txt.data["content"] and 0 < len(content) <= 200, len(content)
            print(f"  File {i+1}: {txt} ({len(content)} chars)")
            print(f"    Starts with: {content[:40]!r}")

    print("\nFile name generation:")
    for i in range(5):
        filename = fake.file_name()
        file_path = fake.file_path()
        assert "." in filename and file_path.startswith("/"), (filename, file_path)

        print(f"  File {i+1}:")
        print(f"    Name: {filename}")
        print(f"    Path: {file_path}")
        print()

except Exception as e:
    print(f"Error in file generation: {e}")
    raise AssertionError(
        f"faker_file example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed")

## Example 2: File System Directory Structure

In [ ]:
from faker import Faker
import faker_file
import os
from datetime import datetime
import random

# Example 2: Generate file system directory structures
fake = Faker()

print("=== Example 2: File System Directory Structure ===")

try:
    # Generate comprehensive directory structure
    directory_structure = {
        'root': '/home/user',
        'subdirectories': {},
        'files': [],
        'total_size': 0,
        'file_count': 0,
        'directory_count': 0
    }
    
    # Common directory types
    common_dirs = ['Documents', 'Downloads', 'Pictures', 'Music', 'Videos', 'Desktop', 'Projects']
    project_dirs = ['src', 'tests', 'docs', 'config', 'data', 'logs', 'backup']
    
    # Generate main directories
    for main_dir in common_dirs[:5]:  # Use first 5 directories
        dir_path = f"{directory_structure['root']}/{main_dir}"
        directory_structure['subdirectories'][main_dir] = {
            'path': dir_path,
            'files': [],
            'subdirectories': [],
            'created': fake.date_time_between(start_date='-2y', end_date='now'),
            'permissions': fake.random_element(elements=['755', '644', '777'])
        }
        directory_structure['directory_count'] += 1
        
        # Add subdirectories
        if main_dir == 'Projects':
            for proj_dir in project_dirs[:3]:  # Add project subdirectories
                subdir_path = f"{dir_path}/{proj_dir}"
                directory_structure['subdirectories'][main_dir]['subdirectories'].append({
                    'name': proj_dir,
                    'path': subdir_path,
                    'created': fake.date_time_between(start_date='-1y', end_date='now'),
                    'permissions': '755'
                })
                directory_structure['directory_count'] += 1
        
        # Generate files for each directory
        file_count_in_dir = fake.random_int(3, 8)
        for i in range(file_count_in_dir):
            # Choose appropriate file types for directory
            if main_dir == 'Pictures':
                extensions = ['.jpg', '.png', '.gif', '.bmp', '.tiff']
            elif main_dir == 'Music':
                extensions = ['.mp3', '.wav', '.flac', '.ogg', '.m4a']
            elif main_dir == 'Videos':
                extensions = ['.mp4', '.avi', '.mkv', '.mov', '.wmv']
            elif main_dir == 'Documents':
                extensions = ['.pdf', '.doc', '.docx', '.txt', '.rtf']
            else:
                extensions = ['.txt', '.json', '.xml', '.csv', '.log']
            
            extension = fake.random_element(elements=extensions)
            filename = f"{fake.word()}_{fake.word()}{extension}"
            file_size = fake.random_int(1024, 50 * 1024 * 1024)  # 1KB to 50MB
            
            file_info = {
                'name': filename,
                'path': f"{dir_path}/{filename}",
                'size': file_size,
                'extension': extension,
                'created': fake.date_time_between(start_date='-1y', end_date='now'),
                'modified': fake.date_time_between(start_date='-30d', end_date='now'),
                'permissions': fake.random_element(elements=['644', '755', '664']),
                'owner': fake.user_name(),
                'group': fake.random_element(elements=['users', 'staff', 'admin'])
            }
            
            directory_structure['subdirectories'][main_dir]['files'].append(file_info)
            directory_structure['files'].append(file_info)
            directory_structure['total_size'] += file_size
            directory_structure['file_count'] += 1
    
    # Display directory structure
    print(f"Directory Structure: {directory_structure['root']}")
    print(f"Total Directories: {directory_structure['directory_count']}")
    print(f"Total Files: {directory_structure['file_count']}")
    print(f"Total Size: {directory_structure['total_size'] / (1024*1024):.2f} MB")
    print()
    
    for dir_name, dir_info in directory_structure['subdirectories'].items():
        print(f"{dir_name}/ ({dir_info['permissions']})")
        print(f"  Path: {dir_info['path']}")
        print(f"  Created: {dir_info['created'].strftime('%Y-%m-%d %H:%M')}")
        print(f"  Files: {len(dir_info['files'])}")
        
        # Show subdirectories
        for subdir in dir_info['subdirectories']:
            print(f"    {subdir['name']}/ ({subdir['permissions']})")
        
        # Show sample files (max 3)
        for file_info in dir_info['files'][:3]:
            size_kb = file_info['size'] / 1024
            print(f"    {file_info['name']} ({size_kb:.1f} KB)")
            print(f"      Owner: {file_info['owner']}:{file_info['group']}")
            print(f"      Modified: {file_info['modified'].strftime('%Y-%m-%d %H:%M')}")
        
        if len(dir_info['files']) > 3:
            print(f"    ... and {len(dir_info['files']) - 3} more files")
        print()
    
    # File type analysis
    print("=== File Type Analysis ===")
    file_types = {}
    for file_info in directory_structure['files']:
        ext = file_info['extension']
        if ext not in file_types:
            file_types[ext] = {'count': 0, 'total_size': 0}
        file_types[ext]['count'] += 1
        file_types[ext]['total_size'] += file_info['size']
    
    for ext, data in sorted(file_types.items(), key=lambda x: x[1]['count'], reverse=True):
        avg_size = data['total_size'] / data['count'] / 1024
        total_size_mb = data['total_size'] / (1024 * 1024)
        print(f"{ext}: {data['count']} files, {total_size_mb:.1f} MB total, {avg_size:.1f} KB avg")

except Exception as e:
    print(f"Error in directory structure generation: {e}")
    raise AssertionError(
        f"faker_file example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed")

## Example 3: File Content and Metadata Generation

In [ ]:
from faker import Faker
import faker_file
import tempfile
import os
import hashlib
import json
from datetime import datetime

# Example 3: Generate file content and comprehensive metadata
fake = Faker()

print("=== Example 3: File Content and Metadata Generation ===")

try:
    # Generate various file types with content
    files_with_content = []
    
    # File type templates
    file_templates = {
        'text': {
            'extensions': ['.txt', '.log', '.md'],
            'content_generator': lambda: fake.text(max_nb_chars=500)
        },
        'config': {
            'extensions': ['.json', '.xml', '.yaml'],
            'content_generator': lambda: json.dumps({
                'app_name': fake.company(),
                'version': f"{fake.random_int(1, 5)}.{fake.random_int(0, 9)}.{fake.random_int(0, 9)}",
                'database_url': fake.url(),
                'debug': fake.boolean(),
                'max_connections': fake.random_int(10, 100)
            }, indent=2)
        },
        'data': {
            'extensions': ['.csv', '.tsv'],
            'content_generator': lambda: '\n'.join([
                'id,name,email,age,city',
                *[f"{fake.random_int(1, 1000)},{fake.name()},{fake.email()},{fake.random_int(18, 65)},{fake.city()}" 
                  for _ in range(fake.random_int(5, 15))]
            ])
        }
    }
    
    for file_type, template in file_templates.items():
        for i in range(2):  # Generate 2 files per type
            extension = fake.random_element(elements=template['extensions'])
            filename = f"{fake.word()}_{fake.word()}{extension}"
            content = template['content_generator']()
            
            # Generate comprehensive metadata
            file_metadata = {
                'basic_info': {
                    'name': filename,
                    'extension': extension,
                    'type': file_type,
                    'size_bytes': len(content.encode('utf-8')),
                    'size_human': f"{len(content.encode('utf-8')) / 1024:.2f} KB"
                },
                'timestamps': {
                    'created': fake.date_time_between(start_date='-1y', end_date='-1m'),
                    'modified': fake.date_time_between(start_date='-1m', end_date='now'),
                    'accessed': fake.date_time_between(start_date='-1w', end_date='now')
                },
                'permissions': {
                    'owner': fake.user_name(),
                    'group': fake.random_element(elements=['users', 'staff', 'admin', 'developers']),
                    'mode': fake.random_element(elements=['644', '755', '664', '600', '777']),
                    'readable': True,
                    'writable': fake.boolean(chance_of_getting_true=80),
                    'executable': extension in ['.sh', '.py', '.exe']
                },
                'content_analysis': {
                    'line_count': content.count('\n') + 1,
                    'word_count': len(content.split()),
                    'character_count': len(content),
                    'encoding': 'utf-8',
                    'has_binary': False,
                    'content_hash': hashlib.md5(content.encode('utf-8')).hexdigest()
                },
                'system_info': {
                    'inode': fake.random_int(100000, 999999),
                    'device': fake.random_element(elements=['/dev/sda1', '/dev/nvme0n1', '/dev/disk1']),
                    'filesystem': fake.random_element(elements=['ext4', 'ntfs', 'apfs', 'xfs']),
                    'backup_status': fake.random_element(elements=['backed_up', 'needs_backup', 'excluded'])
                }
            }
            
            files_with_content.append({
                'content': content,
                'metadata': file_metadata
            })
    
    # Display file information
    for i, file_data in enumerate(files_with_content):
        meta = file_data['metadata']
        print(f"=== File {i+1}: {meta['basic_info']['name']} ===")
        
        # Basic information
        print(f"Type: {meta['basic_info']['type']} ({meta['basic_info']['extension']})")
        print(f"Size: {meta['basic_info']['size_human']} ({meta['basic_info']['size_bytes']} bytes)")
        print(f"Hash: {meta['content_analysis']['content_hash']}")
        
        # Timestamps
        print(f"Created: {meta['timestamps']['created'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Modified: {meta['timestamps']['modified'].strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Last Accessed: {meta['timestamps']['accessed'].strftime('%Y-%m-%d %H:%M:%S')}")
        
        # Permissions
        perms = meta['permissions']
        print(f"Owner: {perms['owner']}:{perms['group']} ({perms['mode']})")
        permission_str = ""
        permission_str += "r" if perms['readable'] else "-"
        permission_str += "w" if perms['writable'] else "-"
        permission_str += "x" if perms['executable'] else "-"
        print(f"Permissions: {permission_str}")
        
        # Content analysis
        content = meta['content_analysis']
        print(f"Content: {content['line_count']} lines, {content['word_count']} words, {content['character_count']} chars")
        
        # System information
        sys_info = meta['system_info']
        print(f"System: inode {sys_info['inode']}, {sys_info['filesystem']} on {sys_info['device']}")
        print(f"Backup: {sys_info['backup_status']}")
        
        # Show content preview (first 100 chars)
        content_preview = file_data['content'][:100]
        if len(file_data['content']) > 100:
            content_preview += "..."
        preview = content_preview.replace(chr(10), '\\n')
        print(f"Preview: {preview}")
        print()
    
    # Generate file operation log
    print("=== File Operation Log ===")
    operations = ['create', 'read', 'write', 'delete', 'copy', 'move', 'chmod', 'chown']
    
    for i in range(8):
        operation = fake.random_element(elements=operations)
        target_file = fake.random_element(elements=files_with_content)['metadata']['basic_info']['name']
        timestamp = fake.date_time_between(start_date='-7d', end_date='now')
        user = fake.user_name()
        success = fake.boolean(chance_of_getting_true=90)
        
        print(f"[{timestamp.strftime('%Y-%m-%d %H:%M:%S')}] {user}: {operation} {target_file} - {'SUCCESS' if success else 'FAILED'}")
    
    # Summary statistics
    total_size = sum(f['metadata']['basic_info']['size_bytes'] for f in files_with_content)
    file_types = {}
    for f in files_with_content:
        ftype = f['metadata']['basic_info']['type']
        file_types[ftype] = file_types.get(ftype, 0) + 1
    
    print(f"\n=== Summary ===")
    print(f"Total Files Generated: {len(files_with_content)}")
    print(f"Total Size: {total_size / 1024:.2f} KB")
    print(f"File Types: {dict(file_types)}")
    print(f"Average File Size: {(total_size / len(files_with_content)) / 1024:.2f} KB")

except Exception as e:
    print(f"Error in file content generation: {e}")
    raise AssertionError(
        f"faker_file example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 completed")

## Example 4: File System Operations and Backup Simulation

In [ ]:
from faker import Faker
import faker_file
import random
from datetime import datetime, timedelta
import json

# Example 4: Generate file system operations and backup scenarios
fake = Faker()

print("=== Example 4: File System Operations and Backup Simulation ===")

try:
    # Generate file system for backup simulation
    file_system = {
        'servers': [],
        'backup_schedules': [],
        'operations_log': [],
        'storage_usage': {}
    }
    
    # Generate servers with file systems
    server_types = ['web', 'database', 'application', 'storage']
    
    for i in range(3):
        server = {
            'server_id': f"SRV{fake.random_number(digits=3)}",
            'hostname': f"{fake.word()}-{fake.random_element(elements=server_types)}-{fake.random_int(1, 99):02d}",
            'ip_address': fake.ipv4(),
            'os': fake.random_element(elements=['Ubuntu 22.04', 'CentOS 8', 'Windows Server 2022', 'RedHat 9']),
            'file_systems': [],
            'total_storage': fake.random_int(100, 2000),  # GB
            'used_storage': 0,
            'backup_enabled': fake.boolean(chance_of_getting_true=85)
        }
        
        # Generate file systems for each server
        fs_types = ['/', '/var', '/home', '/tmp', '/opt']
        
        for fs in fs_types[:fake.random_int(2, 4)]:
            capacity = fake.random_int(10, 500)  # GB
            used = fake.random_int(5, int(capacity * 0.8))
            
            filesystem = {
                'mount_point': fs,
                'type': fake.random_element(elements=['ext4', 'xfs', 'btrfs', 'ntfs']),
                'capacity_gb': capacity,
                'used_gb': used,
                'free_gb': capacity - used,
                'usage_percent': round((used / capacity) * 100, 1),
                'inode_count': fake.random_int(100000, 1000000),
                'inode_used': fake.random_int(50000, 500000),
                'last_check': fake.date_time_between(start_date='-30d', end_date='now')
            }
            
            server['file_systems'].append(filesystem)
            server['used_storage'] += used
        
        file_system['servers'].append(server)
    
    # Generate backup schedules
    backup_types = ['full', 'incremental', 'differential']
    frequencies = ['daily', 'weekly', 'monthly']
    
    for server in file_system['servers']:
        if server['backup_enabled']:
            for i in range(fake.random_int(1, 3)):
                schedule = {
                    'schedule_id': f"BKP{fake.random_number(digits=4)}",
                    'server_id': server['server_id'],
                    'backup_type': fake.random_element(elements=backup_types),
                    'frequency': fake.random_element(elements=frequencies),
                    'time': f"{fake.random_int(0, 23):02d}:{fake.random_int(0, 59):02d}",
                    'retention_days': fake.random_int(7, 365),
                    'destination': f"backup-server-{fake.random_int(1, 5)}.company.com",
                    'compression': fake.boolean(chance_of_getting_true=80),
                    'encryption': fake.boolean(chance_of_getting_true=90),
                    'last_run': fake.date_time_between(start_date='-7d', end_date='now'),
                    'next_run': fake.date_time_between(start_date='now', end_date='+7d'),
                    'status': fake.random_element(elements=['active', 'paused', 'failed', 'completed'])
                }
                file_system['backup_schedules'].append(schedule)
    
    # Generate file operations log
    operations = [
        'file_create', 'file_delete', 'file_modify', 'file_copy', 'file_move',
        'dir_create', 'dir_delete', 'permission_change', 'backup_start', 'backup_complete'
    ]
    
    for i in range(15):
        operation = {
            'timestamp': fake.date_time_between(start_date='-7d', end_date='now'),
            'server_id': fake.random_element(elements=file_system['servers'])['server_id'],
            'operation': fake.random_element(elements=operations),
            'path': f"/{'/'.join([fake.word() for _ in range(fake.random_int(1, 4))])}/{fake.file_name()}",
            'user': fake.user_name(),
            'size_mb': fake.random_int(1, 1000) if 'file' in fake.random_element(elements=operations) else None,
            'duration_ms': fake.random_int(10, 5000),
            'status': fake.random_element(elements=['success', 'failed', 'timeout']),
            'error_code': fake.random_int(1, 999) if fake.boolean(chance_of_getting_true=20) else None
        }
        file_system['operations_log'].append(operation)
    
    # Display server information
    print("=== Server File System Overview ===")
    for server in file_system['servers']:
        print(f"Server: {server['hostname']} ({server['server_id']})")
        print(f"  IP: {server['ip_address']}")
        print(f"  OS: {server['os']}")
        print(f"  Total Storage: {server['total_storage']} GB")
        print(f"  Used Storage: {server['used_storage']} GB")
        print(f"  Backup Enabled: {'Yes' if server['backup_enabled'] else 'No'}")
        
        print(f"  File Systems:")
        for fs in server['file_systems']:
            print(f"    {fs['mount_point']} ({fs['type']}): {fs['used_gb']}/{fs['capacity_gb']} GB ({fs['usage_percent']}% used)")
        print()
    
    # Display backup schedules
    print("=== Backup Schedules ===")
    for schedule in file_system['backup_schedules']:
        server_name = next(s['hostname'] for s in file_system['servers'] if s['server_id'] == schedule['server_id'])
        print(f"Schedule {schedule['schedule_id']}: {server_name}")
        print(f"  Type: {schedule['backup_type']} backup, {schedule['frequency']} at {schedule['time']}")
        print(f"  Retention: {schedule['retention_days']} days")
        print(f"  Destination: {schedule['destination']}")
        print(f"  Features: {'Compressed' if schedule['compression'] else 'Uncompressed'}, {'Encrypted' if schedule['encryption'] else 'Plain'}")
        print(f"  Last Run: {schedule['last_run'].strftime('%Y-%m-%d %H:%M')}")
        print(f"  Status: {schedule['status']}")
        print()
    
    # Display recent operations
    print("=== Recent File Operations ===")
    recent_ops = sorted(file_system['operations_log'], key=lambda x: x['timestamp'], reverse=True)[:10]
    
    for op in recent_ops:
        server_name = next(s['hostname'] for s in file_system['servers'] if s['server_id'] == op['server_id'])
        timestamp_str = op['timestamp'].strftime('%Y-%m-%d %H:%M:%S')
        
        print(f"[{timestamp_str}] {server_name}: {op['operation']}")
        print(f"  Path: {op['path']}")
        print(f"  User: {op['user']}")
        if op['size_mb']:
            print(f"  Size: {op['size_mb']} MB")
        print(f"  Duration: {op['duration_ms']} ms")
        print(f"  Status: {op['status']}")
        if op['error_code']:
            print(f"  Error Code: {op['error_code']}")
        print()
    
    # Generate storage usage statistics
    print("=== Storage Usage Statistics ===")
    total_capacity = sum(server['total_storage'] for server in file_system['servers'])
    total_used = sum(server['used_storage'] for server in file_system['servers'])
    total_free = total_capacity - total_used
    usage_percent = (total_used / total_capacity) * 100
    
    print(f"Overall Storage:")
    print(f"  Total Capacity: {total_capacity:,} GB")
    print(f"  Used Space: {total_used:,} GB")
    print(f"  Free Space: {total_free:,} GB")
    print(f"  Usage: {usage_percent:.1f}%")
    
    # High usage warnings
    high_usage_servers = [s for s in file_system['servers'] 
                         if (s['used_storage'] / s['total_storage']) > 0.8]
    
    if high_usage_servers:
        print(f"\n⚠️  High Usage Alerts ({len(high_usage_servers)} servers):")
        for server in high_usage_servers:
            usage = (server['used_storage'] / server['total_storage']) * 100
            print(f"  {server['hostname']}: {usage:.1f}% usage")
    
    # Backup health summary
    active_backups = [s for s in file_system['backup_schedules'] if s['status'] == 'active']
    failed_backups = [s for s in file_system['backup_schedules'] if s['status'] == 'failed']
    
    print(f"\nBackup Health:")
    print(f"  Total Schedules: {len(file_system['backup_schedules'])}")
    print(f"  Active: {len(active_backups)}")
    print(f"  Failed: {len(failed_backups)}")
    if failed_backups:
        print(f"  ⚠️  Failed backup servers: {', '.join([next(s['hostname'] for s in file_system['servers'] if s['server_id'] == b['server_id']) for b in failed_backups])}")

except Exception as e:
    print(f"Error in file system operations simulation: {e}")
    raise AssertionError(
        f"faker_file example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 completed")

## Example 5: Advanced File Management and Archive System

In [ ]:
from faker import Faker
import faker_file
import random
import tempfile
import json
from datetime import datetime, timedelta
from collections import defaultdict

# Example 5: Advanced file management and archive system simulation
fake = Faker()

print("=== Example 5: Advanced File Management and Archive System ===")

class FileArchiveSystem:
    """Advanced file management and archiving system simulation."""
    
    def __init__(self):
        self.fake = Faker()
        self.file_database = []
        self.archive_policies = []
        self.storage_tiers = []
        self.file_operations = []
        
    def initialize_storage_tiers(self):
        """Initialize different storage tiers."""
        self.storage_tiers = [
            {
                'tier_id': 'HOT',
                'name': 'Hot Storage',
                'cost_per_gb': 0.023,
                'access_time_ms': 1,
                'retention_days': 30,
                'description': 'High-performance SSD storage for frequently accessed files'
            },
            {
                'tier_id': 'WARM',
                'name': 'Warm Storage', 
                'cost_per_gb': 0.012,
                'access_time_ms': 100,
                'retention_days': 90,
                'description': 'Standard storage for occasionally accessed files'
            },
            {
                'tier_id': 'COLD',
                'name': 'Cold Storage',
                'cost_per_gb': 0.004,
                'access_time_ms': 3600000,  # 1 hour
                'retention_days': 2555,  # 7 years
                'description': 'Archive storage for long-term retention'
            },
            {
                'tier_id': 'FROZEN',
                'name': 'Frozen Storage',
                'cost_per_gb': 0.001,
                'access_time_ms': 43200000,  # 12 hours
                'retention_days': 3650,  # 10 years
                'description': 'Deep archive for compliance and backup'
            }
        ]
    
    def generate_file_database(self, count=100):
        """Generate comprehensive file database."""
        file_types = {
            'document': {
                'extensions': ['.pdf', '.doc', '.docx', '.txt', '.rtf'],
                'avg_size_mb': 2.5,
                'access_pattern': 'moderate'
            },
            'image': {
                'extensions': ['.jpg', '.png', '.gif', '.tiff', '.bmp'],
                'avg_size_mb': 5.2,
                'access_pattern': 'low'
            },
            'video': {
                'extensions': ['.mp4', '.avi', '.mkv', '.mov', '.wmv'],
                'avg_size_mb': 250.0,
                'access_pattern': 'very_low'
            },
            'database': {
                'extensions': ['.db', '.sql', '.bak', '.mdb'],
                'avg_size_mb': 150.0,
                'access_pattern': 'high'
            },
            'log': {
                'extensions': ['.log', '.txt', '.out'],
                'avg_size_mb': 0.5,
                'access_pattern': 'moderate'
            }
        }
        
        for i in range(count):
            file_type = self.fake.random_element(elements=list(file_types.keys()))
            type_info = file_types[file_type]
            
            extension = self.fake.random_element(elements=type_info['extensions'])
            base_size = type_info['avg_size_mb']
            actual_size = max(0.1, base_size * random.uniform(0.1, 3.0))
            
            # Determine storage tier based on file age and access pattern
            created_date = self.fake.date_time_between(start_date='-3y', end_date='now')
            days_old = (datetime.now() - created_date).days
            
            if days_old < 30 and type_info['access_pattern'] in ['high', 'moderate']:
                storage_tier = 'HOT'
            elif days_old < 180:
                storage_tier = 'WARM'
            elif days_old < 365:
                storage_tier = 'COLD'
            else:
                storage_tier = 'FROZEN'
            
            file_record = {
                'file_id': f"FILE{self.fake.random_number(digits=8)}",
                'name': f"{self.fake.word()}_{self.fake.word()}{extension}",
                'path': f"/{'/'.join([self.fake.word() for _ in range(self.fake.random_int(1, 5))])}",
                'size_mb': round(actual_size, 2),
                'type': file_type,
                'extension': extension,
                'created': created_date,
                'last_accessed': self.fake.date_time_between(start_date=created_date, end_date='now'),
                'last_modified': self.fake.date_time_between(start_date=created_date, end_date='now'),
                'storage_tier': storage_tier,
                'access_count': self.fake.random_int(0, 1000),
                'checksum': self.fake.sha256(),
                'owner': self.fake.user_name(),
                'department': self.fake.random_element(elements=['IT', 'HR', 'Finance', 'Marketing', 'Operations']),
                'tags': [self.fake.word() for _ in range(self.fake.random_int(0, 4))],
                'encrypted': self.fake.boolean(chance_of_getting_true=30),
                'compliance_class': self.fake.random_element(elements=['public', 'internal', 'confidential', 'restricted'])
            }
            
            self.file_database.append(file_record)
    
    def generate_archive_policies(self):
        """Generate archive policies."""
        policy_templates = [
            {
                'name': 'Document Retention Policy',
                'file_types': ['document'],
                'hot_to_warm_days': 90,
                'warm_to_cold_days': 365,
                'cold_to_frozen_days': 2555,
                'delete_after_days': 3650
            },
            {
                'name': 'Media Archive Policy',
                'file_types': ['image', 'video'],
                'hot_to_warm_days': 30,
                'warm_to_cold_days': 180,
                'cold_to_frozen_days': 730,
                'delete_after_days': None  # Keep forever
            },
            {
                'name': 'Database Backup Policy',
                'file_types': ['database'],
                'hot_to_warm_days': 7,
                'warm_to_cold_days': 90,
                'cold_to_frozen_days': 365,
                'delete_after_days': 2555
            }
        ]
        
        for template in policy_templates:
            policy = {
                'policy_id': f"POL{self.fake.random_number(digits=4)}",
                'name': template['name'],
                'file_types': template['file_types'],
                'transitions': {
                    'hot_to_warm_days': template['hot_to_warm_days'],
                    'warm_to_cold_days': template['warm_to_cold_days'],
                    'cold_to_frozen_days': template['cold_to_frozen_days'],
                    'delete_after_days': template['delete_after_days']
                },
                'created_by': self.fake.name(),
                'created_date': self.fake.date_time_between(start_date='-1y', end_date='now'),
                'active': True,
                'applied_files': len([f for f in self.file_database if f['type'] in template['file_types']])
            }
            self.archive_policies.append(policy)
    
    def generate_analytics_report(self):
        """Generate comprehensive analytics report."""
        
        # Storage tier analysis
        tier_stats = defaultdict(lambda: {'count': 0, 'size_mb': 0, 'cost': 0})
        
        for file_record in self.file_database:
            tier = file_record['storage_tier']
            tier_info = next(t for t in self.storage_tiers if t['tier_id'] == tier)
            
            tier_stats[tier]['count'] += 1
            tier_stats[tier]['size_mb'] += file_record['size_mb']
            tier_stats[tier]['cost'] += file_record['size_mb'] / 1024 * tier_info['cost_per_gb']
        
        # Department analysis
        dept_stats = defaultdict(lambda: {'count': 0, 'size_mb': 0})
        for file_record in self.file_database:
            dept = file_record['department']
            dept_stats[dept]['count'] += 1
            dept_stats[dept]['size_mb'] += file_record['size_mb']
        
        # File type analysis
        type_stats = defaultdict(lambda: {'count': 0, 'size_mb': 0, 'avg_age_days': 0})
        for file_record in self.file_database:
            file_type = file_record['type']
            age_days = (datetime.now() - file_record['created']).days
            
            type_stats[file_type]['count'] += 1
            type_stats[file_type]['size_mb'] += file_record['size_mb']
            type_stats[file_type]['avg_age_days'] = (type_stats[file_type]['avg_age_days'] * 
                                                    (type_stats[file_type]['count'] - 1) + age_days) / type_stats[file_type]['count']
        
        return {
            'tier_stats': dict(tier_stats),
            'dept_stats': dict(dept_stats),
            'type_stats': dict(type_stats),
            'total_files': len(self.file_database),
            'total_size_gb': sum(f['size_mb'] for f in self.file_database) / 1024,
            'total_monthly_cost': sum(tier_stats[tier]['cost'] for tier in tier_stats)
        }

try:
    # Initialize the file archive system
    archive_system = FileArchiveSystem()
    
    print("Initializing file archive system...")
    archive_system.initialize_storage_tiers()
    archive_system.generate_file_database(75)  # Generate 75 files for demo
    archive_system.generate_archive_policies()
    
    # Display storage tiers
    print("=== Storage Tiers Configuration ===")
    for tier in archive_system.storage_tiers:
        print(f"{tier['name']} ({tier['tier_id']}):")
        print(f"  Cost: ${tier['cost_per_gb']:.3f} per GB/month")
        print(f"  Access Time: {tier['access_time_ms']:,} ms")
        print(f"  Default Retention: {tier['retention_days']} days")
        print(f"  Description: {tier['description']}")
        print()
    
    # Display archive policies
    print("=== Archive Policies ===")
    for policy in archive_system.archive_policies:
        print(f"Policy: {policy['name']} ({policy['policy_id']})")
        print(f"  File Types: {', '.join(policy['file_types'])}")
        print(f"  Transitions:")
        transitions = policy['transitions']
        print(f"    Hot → Warm: {transitions['hot_to_warm_days']} days")
        print(f"    Warm → Cold: {transitions['warm_to_cold_days']} days")
        print(f"    Cold → Frozen: {transitions['cold_to_frozen_days']} days")
        if transitions['delete_after_days']:
            print(f"    Delete After: {transitions['delete_after_days']} days")
        else:
            print(f"    Delete After: Never")
        print(f"  Applied to {policy['applied_files']} files")
        print()
    
    # Generate and display analytics
    print("=== File Archive Analytics ===")
    analytics = archive_system.generate_analytics_report()
    
    print(f"System Overview:")
    print(f"  Total Files: {analytics['total_files']:,}")
    print(f"  Total Size: {analytics['total_size_gb']:.2f} GB")
    print(f"  Monthly Storage Cost: ${analytics['total_monthly_cost']:.2f}")
    
    print(f"\nStorage Tier Distribution:")
    for tier_id, stats in analytics['tier_stats'].items():
        tier_name = next(t['name'] for t in archive_system.storage_tiers if t['tier_id'] == tier_id)
        percentage = (stats['count'] / analytics['total_files']) * 100
        print(f"  {tier_name}: {stats['count']} files ({percentage:.1f}%), {stats['size_mb']/1024:.2f} GB, ${stats['cost']:.2f}/month")
    
    print(f"\nDepartment Usage:")
    sorted_depts = sorted(analytics['dept_stats'].items(), key=lambda x: x[1]['size_mb'], reverse=True)
    for dept, stats in sorted_depts:
        print(f"  {dept}: {stats['count']} files, {stats['size_mb']/1024:.2f} GB")
    
    print(f"\nFile Type Analysis:")
    for file_type, stats in analytics['type_stats'].items():
        print(f"  {file_type.title()}: {stats['count']} files, {stats['size_mb']/1024:.2f} GB, avg age {stats['avg_age_days']:.0f} days")
    
    # Storage optimization recommendations
    print(f"\n=== Storage Optimization Recommendations ===")
    
    # Find files that can be moved to cheaper tiers
    optimization_candidates = []
    for file_record in archive_system.file_database:
        days_old = (datetime.now() - file_record['created']).days
        days_since_access = (datetime.now() - file_record['last_accessed']).days
        
        if file_record['storage_tier'] == 'HOT' and days_since_access > 30:
            optimization_candidates.append(f"Move {file_record['name']} to Warm storage (not accessed for {days_since_access} days)")
        elif file_record['storage_tier'] == 'WARM' and days_old > 180:
            optimization_candidates.append(f"Move {file_record['name']} to Cold storage (created {days_old} days ago)")
    
    print(f"Found {len(optimization_candidates)} optimization opportunities:")
    for recommendation in optimization_candidates[:5]:  # Show first 5
        print(f"  • {recommendation}")
    
    if len(optimization_candidates) > 5:
        print(f"  ... and {len(optimization_candidates) - 5} more opportunities")
    
    # Calculate potential savings
    potential_savings = 0
    for file_record in archive_system.file_database:
        if file_record['storage_tier'] == 'HOT' and (datetime.now() - file_record['last_accessed']).days > 30:
            hot_cost = (file_record['size_mb'] / 1024) * 0.023
            warm_cost = (file_record['size_mb'] / 1024) * 0.012
            potential_savings += (hot_cost - warm_cost)
    
    print(f"\nPotential Monthly Savings: ${potential_savings:.2f}")
    
except Exception as e:
    print(f"Error in file archive system: {e}")
    raise AssertionError(
        f"faker_file example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 completed")